# This Notebook contains the code to generate the json file for the search function of the navigation bar

> #### This is a basic implementation of making a pages.json dictionary for searching

In [1]:
import os
import json
from bs4 import BeautifulSoup

def extract_title(file_path):
    """Extract <title> from HTML file, fallback to filename."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "html.parser")
            title_tag = soup.find("title")
            if title_tag and title_tag.text.strip():
                return title_tag.text.strip()
    except Exception as e:
        print(f"Could not parse {file_path}: {e}")
    return os.path.basename(file_path)

def main():
    root_dir = "."  # repo root
    pages = []

    for subdir, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith(".html"):
                file_path = os.path.join(subdir, file)
                rel_path = os.path.relpath(file_path, root_dir).replace("\\", "/")

                title = extract_title(file_path)
                pages.append({
                    "title": title,
                    "url": rel_path,
                    "content": ""  # optional: could scrape <p> text if desired
                })

    with open("pages.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, indent=2, ensure_ascii=False)

    print(f"Generated pages.json with {len(pages)} entries")

if __name__ == "__main__":
    main()


Generated pages.json with 710 entries


> #### This improves the pages.json dictionary for searching by adding content from the entire website

In [3]:
import os
import json
from bs4 import BeautifulSoup

BASE_DIR = "."

pages = []

for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if file.endswith(".html"):
            path = os.path.join(root, file)
            rel_path = os.path.relpath(path, BASE_DIR)
            
            with open(path, "r", encoding="utf-8") as f:
                soup = BeautifulSoup(f, "html.parser")
                
                # Title
                title = os.path.splitext(file)[0].replace("_", " ").title()
                
                # Remove scripts/styles
                for s in soup(["script", "style"]):
                    s.extract()
                
                # Extract visible text
                text = soup.get_text(" ", strip=True)
                
                # Use snippet for preview
                snippet = text[:300] + "..." if len(text) > 300 else text
                
                pages.append({
                    "title": title,
                    "url": rel_path.replace("\\", "/"),
                    "content": text,
                    "snippet": snippet
                })

with open("pages.json", "w", encoding="utf-8") as out:
    json.dump(pages, out, indent=2, ensure_ascii=False)

print("✅ pages.json generated with", len(pages), "pages")


✅ pages.json generated with 710 pages


> #### This further improves the pages.json dictionary for searching by adding content from the relevant directories

In [6]:
import os
import json
from bs4 import BeautifulSoup

BASE_DIR = "."

# Folders to skip
SKIP_FOLDERS = {
    "TVSaffiliations/scripts",
    "scripts",
    "RubinRhapsodies",
    "old"
}

pages = []

for root, dirs, files in os.walk(BASE_DIR):
    # Get relative path from BASE_DIR
    rel_root = os.path.relpath(root, BASE_DIR)
    
    # Normalize path separators for comparison
    rel_root_normalized = rel_root.replace("\\", "/")
    
    # Skip if current directory matches any skip folder
    should_skip = False
    for skip_folder in SKIP_FOLDERS:
        # Check if we're in the skip folder or a subdirectory of it
        if rel_root_normalized == skip_folder or rel_root_normalized.startswith(skip_folder + "/"):
            should_skip = True
            break
    
    if should_skip:
        # Clear dirs list to prevent os.walk from descending into subdirectories
        dirs[:] = []
        continue
    
    for file in files:
        if file.endswith(".html"):
            path = os.path.join(root, file)
            rel_path = os.path.relpath(path, BASE_DIR)
            
            with open(path, "r", encoding="utf-8") as f:
                soup = BeautifulSoup(f, "html.parser")
                
                # Title
                title = os.path.splitext(file)[0].replace("_", " ").title()
                
                # Remove scripts/styles
                for s in soup(["script", "style"]):
                    s.extract()
                
                # Extract visible text
                text = soup.get_text(" ", strip=True)
                
                # Use snippet for preview
                snippet = text[:300] + "..." if len(text) > 300 else text
                
                pages.append({
                    "title": title,
                    "url": rel_path.replace("\\", "/"),
                    "content": text,
                    "snippet": snippet
                })

with open("pages.json", "w", encoding="utf-8") as out:
    json.dump(pages, out, indent=2, ensure_ascii=False)

print("pages.json generated with", len(pages), "pages")

pages.json generated with 51 pages
